# T03 — CellRank 2: Trajectory Inference & Cell Fate Mapping

**Tools:** `cellrank` (v2), `scvelo`  
**Dataset:** Mouse pancreatic endocrinogenesis (E15.5 developmental time course)  
**Papers:** [Weiler et al. 2024, Nature Methods](https://doi.org/10.1038/s41592-024-02303-9) · [Bergen et al. 2020, Nature Biotechnology](https://doi.org/10.1038/s41587-020-0591-3)

---

## What is trajectory inference?

In a snapshot of developing cells, we observe a mixture of cells at different stages. Trajectory inference reconstructs the **pseudotime** ordering and identifies which cells are progenitors vs. mature cell types.

## What is CellRank?

CellRank frames trajectory inference as a **Markov chain problem**: each cell has a probability of transitioning to each of its neighbors. The stationary distribution of this chain identifies **terminal states** (attractors = mature cell types).

CellRank 2 is modular: you can combine multiple **kernels** (sources of directionality):

| Kernel | Signal source | When to use |
|--------|--------------|-------------|
| `VelocityKernel` | RNA velocity (spliced/unspliced ratio) | Time course without timestamps |
| `ConnectivityKernel` | Gene expression similarity | Background regularization |
| `CytoTRACEKernel` | Transcriptional diversity (high = progenitor) | No velocity data |
| `PseudotimeKernel` | External pseudotime or DPT | When you have prior trajectory |
| `RealTimeKernel` | Experimental time stamps + OT | Time-resolved experiments |

## RNA Velocity (scVelo)

Cells are simultaneously sequencing **nascent (unspliced)** and **mature (spliced)** mRNA. If a gene's unspliced fraction is *higher* than expected at steady state → the cell is *upregulating* that gene → it's moving in that direction in expression space. This is the velocity vector.

scVelo (the Theis Lab's velocity tool) estimates these vectors from the spliced/unspliced ratio.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import cellrank as cr
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')
scv.settings.verbosity = 2
scv.settings.set_figure_params('scvelo', dpi=80)
print(f'scvelo {scv.__version__}  |  cellrank {cr.__version__}')

## 1. Load Pancreas Developmental Dataset

In [ ]:
# Mouse pancreatic endocrinogenesis — Bastidas-Ponce et al. 2019
# ~3,696 cells capturing the E15.5 developmental trajectory
# Contains spliced + unspliced counts in separate layers
adata = scv.datasets.pancreas()
print(adata)
print('\nLayers:', list(adata.layers.keys()))
print('Cell type distribution:')
print(adata.obs['clusters'].value_counts())

In [ ]:
# Quick UMAP colored by annotated cell types
sc.pl.umap(adata, color='clusters', legend_loc='on data',
           title='Pancreas E15.5 — annotated clusters')

## 2. RNA Velocity with scVelo

scVelo needs:
1. `adata.layers['spliced']` — mature mRNA counts
2. `adata.layers['unspliced']` — nascent mRNA counts

The workflow: filter → normalize per-gene → first-order moments (mean + variance over neighbors) → fit kinetic model → compute velocity graph

In [ ]:
# Preprocessing for velocity
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
# Computes first and second order moments (mean + uncentered variance)
# over kNN neighborhood — required for velocity estimation
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

In [ ]:
# Recover dynamics (EM algorithm fitting splicing kinetics per gene)
# mode='stochastic' is faster; 'dynamical' is more accurate but slower
scv.tl.recover_dynamics(adata, n_jobs=4)
scv.tl.velocity(adata, mode='dynamical')

# Build velocity graph: probability of transitioning to each neighbor
# based on velocity direction cosine similarity
scv.tl.velocity_graph(adata)

In [ ]:
# Visualize RNA velocity as stream plot on UMAP
scv.pl.velocity_embedding_stream(
    adata,
    basis='umap',
    color='clusters',
    title='RNA velocity stream',
    figsize=(6, 5),
)
# The arrows show the predicted direction of transcriptional change

In [ ]:
# Latent time — scVelo's pseudotime inferred from velocity
scv.tl.latent_time(adata)
scv.pl.scatter(
    adata,
    color='latent_time',
    color_map='gnuplot',
    size=80,
    title='scVelo latent time',
)

## 3. CellRank 2: Markov Chain Fate Mapping

Now we use CellRank to formalize the trajectory into:
1. **Terminal states** (macrostates that cells converge to)
2. **Initial states** (source populations)
3. **Fate probabilities** (per cell, probability of reaching each terminal state)
4. **Driver genes** (genes correlated with fate probability)

In [ ]:
from cellrank.kernels import VelocityKernel, ConnectivityKernel
from cellrank.estimators import GPCCA

# VelocityKernel: uses the scVelo velocity graph
# Each entry T[i,j] = probability of cell i transitioning to cell j
vk = VelocityKernel(adata)
vk.compute_transition_matrix()
print('VelocityKernel transition matrix:', vk.transition_matrix.shape)

In [ ]:
# ConnectivityKernel: symmetric diffusion kernel over gene expr neighbors
# Used to regularize the velocity kernel (smooth out noise)
ck = ConnectivityKernel(adata)
ck.compute_transition_matrix()

# Combined kernel: 80% velocity signal, 20% connectivity regularization
combined = 0.8 * vk + 0.2 * ck
combined.compute_transition_matrix()
print('Combined kernel ready')

In [ ]:
# GPCCA: Generalized Perron Cluster Cluster Analysis
# Identifies macrostates as nearly-absorbing sets in the Markov chain
g = GPCCA(combined)

# fit() computes the Schur decomposition of the transition matrix
# n_states: range to search for optimal macrostate count
g.fit(n_states=[3, 10], cluster_key='clusters')
g.plot_macrostates(which='all', basis='umap', legend_loc='on data')

In [ ]:
# Assign terminal states (where the Markov chain absorbs = mature cell types)
g.predict_terminal_states(method='top_n', n_states=3)
g.plot_macrostates(which='terminal', basis='umap', legend_loc='on data')

In [ ]:
# Compute fate probabilities
# Each cell gets a probability vector summing to 1 over terminal states
g.compute_fate_probabilities()
print('Fate probabilities stored in adata.obsm["lineages_fwd"]')
print(adata.obsm['lineages_fwd'])  # FateProbabilities object

In [ ]:
# Plot fate probabilities on UMAP
g.plot_fate_probabilities(same_plot=False, basis='umap')

## 4. CytoTRACEKernel: Trajectory Without Velocity

No spliced/unspliced data? CytoTRACE uses a simpler heuristic: cells with more diverse transcriptomes (more genes expressed) tend to be less differentiated. This is surprisingly effective.

In [ ]:
from cellrank.kernels import CytoTRACEKernel

ctk = CytoTRACEKernel(adata)
ctk.compute_cytotrace()
ctk.compute_transition_matrix(threshold_scheme='soft', nu=0.5)

# Compare CytoTRACE score vs latent time
sc.pl.umap(
    adata,
    color=['ct_score', 'latent_time'],
    color_map='gnuplot',
    title=['CytoTRACE score (high=progenitor)', 'scVelo latent time'],
)

## 5. Driver Genes

Which genes are most correlated with fate probability toward a specific lineage? These are candidate **fate drivers**.

In [ ]:
# Correlate gene expression with fate probability toward each terminal state
# Returns ranked DataFrame of genes
for terminal in g.terminal_states.cat.categories:
    try:
        drivers = g.compute_lineage_drivers(
            lineages=terminal,
            cluster_key='clusters',
            use_raw=False,
        )
        print(f'\nTop drivers for {terminal}:')
        print(drivers.head(5)[['corr', 'pval']])
    except Exception as e:
        print(f'Could not compute drivers for {terminal}: {e}')

---
## Summary

```python
# Core CellRank 2 workflow
from cellrank.kernels import VelocityKernel, ConnectivityKernel, CytoTRACEKernel
from cellrank.estimators import GPCCA

# 1. Build kernel(s)
vk = VelocityKernel(adata).compute_transition_matrix()
ck = ConnectivityKernel(adata).compute_transition_matrix()
combined = 0.8 * vk + 0.2 * ck

# 2. Fit GPCCA estimator
g = GPCCA(combined)
g.fit(n_states=[3, 10], cluster_key='clusters')
g.predict_terminal_states()

# 3. Fate probabilities and drivers
g.compute_fate_probabilities()
g.compute_lineage_drivers(lineages='Alpha cell')
```

| CellRank object | What it is |
|-----------------|------------|
| `VelocityKernel` | Directed graph from RNA velocity |
| `CytoTRACEKernel` | Directed graph from transcriptional diversity |
| `ConnectivityKernel` | Undirected similarity graph (regularization) |
| `GPCCA` | Estimator that finds macrostates + fate probabilities |
| `adata.obsm['lineages_fwd']` | Per-cell fate probability matrix |

**Next:** T04 — Muon + MOFA+ for multi-modal analysis.